## Notebook 01: Data Loading & Exploration

**Research question**: Did Australian reforestation and rewilding sites
show measurable NDVI recovery between 2015–2024, and does proximity
to intervention sites predict vegetation gain?

Data sources:
  - MODIS MOD13Q1 NDVI (via GEE export → data/processed/)
  - Hansen tree cover change (GeoTIFF, via GEE export)
  - GBIF plant species occurrences (live API)
  - Study site boundaries (GeoJSON, manually defined)

Outputs:
  - figures/site_map.png
  - figures/ndvi_raw_overview.png
  - data/processed/ndvi_clean.parquet

In [ ]:
# Imports & Paths
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import json, requests
from pathlib import Path

BASE    = Path("..")          # Repo root
DATA    = BASE / "data"
FIG     = BASE / "figures"
FIG_MAP = FIG / "maps"
FIG_MAP.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
PALETTE = {"treated": "#1a9850", "control": "#d73027"}
print("Paths OK:", DATA.exists(), FIG.exists())

In [ ]:
# Load NDVI CSV from GEE export
# ============================================================
# After running GEE scripts: download from Google Drive →
# place at data/processed/ndvi_modis_all_sites.csv

ndvi_path = DATA / "processed" / "ndvi_modis_all_sites.csv"

# Fallback: synthetic data if GEE export not yet available
if ndvi_path.exists():
    ndvi = pd.read_csv(ndvi_path, parse_dates=["date"])
    print(f"✓ Loaded real MODIS data: {ndvi.shape}")
else:
    print("⚠  GEE export not found — using synthetic proxy data")
    print("   Register at https://earthengine.google.com and run gee/ndvi_extraction.js")
    # [Insert Phase 1 synthetic generator here from previous conversation]
    ndvi = generate_all_synthetic()   # placeholder call

# ── Basic cleaning ──
ndvi = ndvi.dropna(subset=["ndvi_mean"])
ndvi["ndvi_mean"] = ndvi["ndvi_mean"].clip(0, 1)
ndvi["date"] = pd.to_datetime(ndvi["date"])
ndvi = ndvi.sort_values(["site_id", "date"]).reset_index(drop=True)
print(ndvi.dtypes)
print(ndvi.head())